# v3.3 iron-man run — the locked version against BC, self-hosted klein on an A100

One model. Every klein call is timed; `run/meta/cost.json` totals the run against the hourly rate you set below and beside the fal-equivalent price. Run cells in order. **Runtime → A100.**

In [ ]:
# 1 · settings
A100_USD_PER_HOUR = 1.20      # what this runtime costs you per hour; edit. Colab Pro A100 ≈ 11.8 CU/h at $0.10/CU.
SEEDS = [46, 47, 48]          # the version is scored at more than one seed; 46 is the seed every V3 run used
ARMS = ("BC", "V")
LIMIT = None                  # e.g. 20 for a partial run; None = the whole matrix
DRIVE_PROJECT_DIR = "Side projects and shi"   # exact Drive folder name; the UI truncates it

In [ ]:
# 2 · Drive: find the HF cache that actually holds klein, and a place to keep the zip
import os
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'
BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
KLEIN = 'models--black-forest-labs--FLUX.2-klein-4B'
candidates = [os.path.join(BASE, 'tryon_models', 'hf_cache'),   # where V2's notebook put it
              os.path.join(MYDRIVE, 'hf_cache'),                 # a root-level hf_cache, if that is where it lives
              os.path.join(BASE, 'hf_cache')]
found = [c for c in candidates if os.path.isdir(os.path.join(c, 'hub', KLEIN))]
HF_HOME = found[0] if found else candidates[0]
os.environ['HF_HOME'] = HF_HOME                                   # must be set before any diffusers import
os.environ['V3_MODEL_DIR'] = os.path.join(BASE if os.path.isdir(BASE) else MYDRIVE, 'v3_models')
os.makedirs(HF_HOME, exist_ok=True); os.makedirs(os.environ['V3_MODEL_DIR'], exist_ok=True)
for c in candidates:
    print(('  klein here  ' if c in found else '  -           ') + c)
print('HF_HOME =', HF_HOME, '(klein cached)' if found else '(no cached klein found - cell 4 downloads ~13 GB here once)')
if len(found) > 1: print('NOTE: klein is cached in more than one place; the first is used. The others are duplicates you can delete.')

In [ ]:
# 3 · install and unpack the bundle (upload v33_ironman_bundle.zip to /content first)
!pip -q install -U diffusers transformers accelerate sentencepiece protobuf mediapipe onnxruntime-gpu opencv-python-headless
import os; assert os.path.exists('/content/v33_ironman_bundle.zip'), 'upload v33_ironman_bundle.zip to /content (Files pane)'
!cd /content && unzip -qo v33_ironman_bundle.zip -d ironman
%cd /content/ironman
import onnxruntime as ort, torch
print('onnxruntime providers:', ort.get_available_providers(), '- BiRefNet needs CUDAExecutionProvider here; on CPU each crop is ~50 s')
print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
# 4 · weights: load once (from the Drive cache; downloads there if absent) and time it
import sys; sys.path.insert(0, 'lib')
import klein_local as K
K.load(); K.info()

In [ ]:
# 5 · one pair first — check run/gen has both arms before spending on the matrix
import run_ironman as R
R.main('matrix.csv', 'testset', limit=1, seeds=SEEDS[:1], arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
import os; print(sorted(os.listdir('run/gen')))

In [ ]:
# 6 · the whole matrix, every seed. Resumable: re-run this cell after any interruption.
R.main('matrix.csv', 'testset', limit=LIMIT, seeds=SEEDS, arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
import json; print(json.dumps(json.load(open('run/meta/cost.json')), indent=1))

In [ ]:
# 7 · zip the evidence and copy it to Drive; download from there or from the Files pane
import shutil, time
name = f"v33_ironman_run_{time.strftime('%Y%m%d_%H%M')}"
shutil.make_archive(f'/content/{name}', 'zip', 'run')
dst = os.path.join(BASE, 'v3_runs', name + '.zip'); os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(f'/content/{name}.zip', dst)
print('zip:', dst)
print('then locally:  python3 v3/build/ironman_page.py', name + '.zip')